|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 0:</h2>|<h1>From a Program to a Model<h1>|
|<h2>Section:</h2>|<h1>How to measure<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: measure it yourself<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

# Measure it yourself

`mysteries.py`, next to this notebook, has three functions. Each takes a size
`n` and returns a function to time. Do not open the file. Find the shape of
each function from the outside, with measurements only: `flat`, `linear` or
`quadratic`.

This is the situation of your real work: a system that you did not write,
and a question about how it grows.

In [ ]:
# find the repo root. The directory you start from does not matter.
import sys
from pathlib import Path
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
            if (folder/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import time
import numpy as np
import torch
import matplotlib.pyplot as plt
import cudalib

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

sys.path.insert(0, str(ROOT/'course/Part0_FromAProgramToAModel/4_measure'))
import hashlib
from mysteries import mystery_1, mystery_2, mystery_3

# Exercise 1: an honest stopwatch

**Given:** a GPU function with no arguments, the three rules of the first
notebook of this section, and `cudalib.bench_ms` to compare with.

**Asked:** `honest_ms(function)`, the milliseconds of one call. You know that
it is correct when the check prints "your stopwatch is honest".

Use the three rules:

1. Warm up: call `function` 3 times first.
2. Wait for the GPU before you read the time.
3. Repeat: time 5 rounds of 10 calls, and return the **median** of the
   milliseconds for one call.

In [ ]:
def honest_ms(function):
    """Milliseconds for one call of `function`: warmed up, waited for, repeated."""
    for _ in range(3):
        function()
    torch.cuda.synchronize()
    rounds = []
    for _ in range(5):
        start = time.perf_counter()
        for _ in range(10):
            function()
        torch.cuda.synchronize()
        rounds.append((time.perf_counter() - start) * 1e3 / 10)
    return float(np.median(rounds))

In [ ]:
# The check: your stopwatch must agree with the stopwatch of the course.
work = torch.randn(4096, 4096, device='cuda', dtype=torch.bfloat16)
yours = honest_ms(lambda: work @ work)
course = cudalib.bench_ms(lambda: work @ work, iters=10, warmup=3, best_of=5)
print(f'yours: {yours:.3f} ms   cudalib.bench_ms: {course:.3f} ms')
assert yours is not None, 'honest_ms returned nothing'
assert 0.7 < yours / course < 1.5, 'too far from cudalib.bench_ms. Did you wait for the GPU?'
print('your stopwatch is honest')

# Exercise 2: the shape of each mystery

**Given:** three functions that you cannot read, your `honest_ms`, and three
sizes.

**Asked:** the shape of each function: `'flat'`, `'linear'` or `'quadratic'`.
You know that you are correct when the check prints `correct` three times.

Time each mystery at `n = 4096`, `8192` and `16384`. Compute the doubling
ratios. Then write the shape of each one: `'flat'`, `'linear'` or
`'quadratic'`.

Read the ratio at the large end. Remember the trap of the last notebook: at
small sizes, a fixed cost can hide the real shape.

In [ ]:
SIZES = [4096, 8192, 16384]
for name, mystery in [('mystery_1', mystery_1), ('mystery_2', mystery_2), ('mystery_3', mystery_3)]:
    times = [honest_ms(mystery(size)) for size in SIZES]
    ms_2n_over_n = [round(later / earlier, 2) for earlier, later in zip(times, times[1:])]
    print(f'{name}: ' + '  '.join(f'{mystery_ms:.3f} ms' for mystery_ms in times) + f'   ratios {ms_2n_over_n}')

In [ ]:
SHAPES = {
    'mystery_1': 'quadratic',
    'mystery_2': 'flat',
    'mystery_3': 'linear',
}

In [ ]:
# The check. The answers are stored as hashes.
EXPECTED = {'mystery_1': '51aabdf60e20', 'mystery_2': 'bde28bbc2dea', 'mystery_3': 'b9ee0a6e437d'}
for name, shape in SHAPES.items():
    ok = hashlib.sha256(f'{name}:{shape.strip().lower()}'.encode()).hexdigest()[:12] == EXPECTED[name]
    print(f'{name}: {"correct" if ok else "not yet"}')

### Why

- **mystery_1** is quadratic. It computes the distance between every pair of
  `n` points, and it writes an `n x n` result. At small `n` the ratio can be
  larger than 4: the result fits in the cache of the GPU at small sizes, and
  not at large ones. That is a second shape inside the first.
- **mystery_2** is flat. Almost all of its time is a fixed read of 256 MB. The
  part that depends on `n` is too small to see at these sizes.
- **mystery_3** is linear. It reads `n x 65536` numbers one time.

Now open `mysteries.py`, and check your reading against the code.

### After the check

You found the shape of three systems from the outside, with measurements
only. That is the whole of move 4. The course uses it on a transformer in
Part 1, on a scheduler in Part 2, and on your own engine in Part 8.